[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/huggingface-nlp-certified/notebooks/day-10-summarization-translation.ipynb#scrollTo=a1b2c3d4)

---
# Day 10 · Summarization and Translation — Seq2Seq Models and Generation Configs
**certified-journeys / huggingface-nlp-certified** · Day 10 · Seq2Seq Tasks

> **Goal for today:** Use BART and T5 for abstractive summarization (including fine-tuning with `Seq2SeqTrainer`) and Helsinki-NLP models for translation, and compare greedy vs. beam-search ROUGE-L scores.


In [ ]:
%pip install -q transformers datasets evaluate rouge_score sentencepiece sacrebleu accelerate


## Step 1 · Summarization task overview — what is seq2seq?

Summarization is an **encoder-decoder** (seq2seq) task: the encoder reads the full document and the decoder generates a shorter output auto-regressively.

| Architecture | Example models | Best for |
|---|---|---|
| Encoder-only | BERT, RoBERTa | Classification, NER |
| Decoder-only | GPT-2, LLaMA | Open-ended generation |
| Encoder-decoder | BART, T5, Pegasus | Summarization, translation, QA |

Key generation hyperparameters to control output quality:
- **`num_beams`** — beam width; higher = better but slower
- **`length_penalty`** — `> 1.0` favours longer outputs, `< 1.0` shorter
- **`early_stopping`** — stop beams once EOS is reached on all beams
- **`max_new_tokens`** — hard cap on generated length

Official guide: [HF Summarization Task Guide](https://huggingface.co/docs/transformers/tasks/summarization)


In [ ]:
from transformers import pipeline

# Load BART-large-CNN — a BART model fine-tuned on CNN/DailyMail
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=-1,  # use CPU; set to 0 for GPU
)

article = """
Scientists at NASA's Jet Propulsion Laboratory have confirmed the discovery of
a planet outside our solar system that shows signs of a water-rich atmosphere.
The exoplanet, designated K2-18b, orbits a red dwarf star about 120 light-years
from Earth in the constellation Leo. Using the James Webb Space Telescope,
researchers detected carbon dioxide and methane in the planet's atmosphere,
a chemical combination that could indicate the presence of a liquid ocean beneath
a hydrogen-rich atmosphere — a type of world scientists call a 'Hycean planet'.
The findings were published in the journal Nature Astronomy and represent one of
the most promising leads in the search for extraterrestrial life.
"""

result = summarizer(article, max_length=80, min_length=30, do_sample=False)
print("Summary:", result[0]["summary_text"])


### What just happened?
- `pipeline('summarization')` downloads BART's tokenizer and weights automatically.
- `do_sample=False` forces **greedy** decoding — deterministic but not always optimal.
- `max_length` and `min_length` constrain token counts of the *generated* summary, not the input.
- **BART-large-CNN** was pre-trained with a denoising objective then fine-tuned on CNN/DailyMail — it already knows newspaper-style summarization without any extra training from us.


## Step 2 · Load and preprocess `cnn_dailymail` for fine-tuning

We'll fine-tune **t5-small** (60 M params) on a subset of CNN/DailyMail. T5 treats every task as text-to-text by prepending a task prefix, e.g. `"summarize: <article>"`.

Dataset columns we need:
| Column | Content |
|---|---|
| `article` | Full news article (input) |
| `highlights` | Multi-sentence summary (target) |

We truncate articles to 512 tokens and summaries to 128 tokens to keep Colab training fast.


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_CKPT = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

# Use a small slice for fast iteration — remove [:500] for a full run
raw = load_dataset("cnn_dailymail", "3.0.0", split="train[:500]")
val_raw = load_dataset("cnn_dailymail", "3.0.0", split="validation[:100]")

PREFIX = "summarize: "  # T5 task prefix
MAX_INPUT = 512
MAX_TARGET = 128

def preprocess(batch):
    inputs = [PREFIX + doc for doc in batch["article"]]
    model_inputs = tokenizer(
        inputs, max_length=MAX_INPUT, truncation=True, padding="max_length"
    )
    # Encode targets separately using text_target parameter
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["highlights"],
            max_length=MAX_TARGET,
            truncation=True,
            padding="max_length",
        )
    # Replace padding token id with -100 so loss ignores pad tokens
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = raw.map(preprocess, batched=True, remove_columns=raw.column_names)
tokenized_val   = val_raw.map(preprocess, batched=True, remove_columns=val_raw.column_names)

print("Train size:", len(tokenized_train))
print("Val size  :", len(tokenized_val))
print("Sample keys:", tokenized_train[0].keys())


### What just happened?
- We prepend `"summarize: "` so T5 knows which of its multi-task heads to activate.
- **Labels padded with -100** tell the cross-entropy loss to skip those positions — without this, the model would try to learn to predict padding tokens.
- `as_target_tokenizer()` is the T5-safe way to encode targets; it applies the correct special tokens for decoder input.
- `remove_columns` strips the raw string columns so the dataset only contains tensors.


## Step 3 · Fine-tune T5-small with `Seq2SeqTrainer`

`Seq2SeqTrainer` extends the standard `Trainer` with one key addition:
**`predict_with_generate=True`** — when evaluating, it runs `model.generate()` instead of a raw forward pass, giving you actual decoded text for ROUGE scoring.

Without this flag, `compute_metrics` receives raw logits and you cannot compute ROUGE.

Generation config we'll set in `Seq2SeqTrainingArguments`:

| Param | Value | Reason |
|---|---|---|
| `num_beams` | 4 | Wider beam → better summaries |
| `length_penalty` | 2.0 | Rewards longer, more complete summaries |
| `early_stopping` | True | Stop when all beams reach EOS |


In [ ]:
import evaluate
import numpy as np
from transformers import (
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

rouge = evaluate.load("rouge")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CKPT)

# Pad the inputs to the same length within a batch
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # predictions are token ids from generate(); decode them
    decoded_preds  = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 with pad id before decoding labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # ROUGE expects newline-separated sentences
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v, 4) for k, v in result.items()}

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-small-cnn",
    num_train_epochs=1,          # increase to 3–5 for a real run
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=True,  # CRITICAL for seq2seq metrics
    generation_num_beams=4,      # beam search during eval
    # length_penalty and early_stopping via GenerationConfig
    fp16=False,                  # set True on GPU for speed
)

# Set generation config directly on the model
model.generation_config.num_beams = 4
model.generation_config.length_penalty = 2.0
model.generation_config.early_stopping = True

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting training (1 epoch on 500 examples — expect ~2 min on CPU)…")
trainer.train()
print("Done!")


### What just happened?
- `Seq2SeqTrainer` ran a standard training loop, but at evaluation time it called `model.generate()` using our beam-search config — that's what `predict_with_generate=True` enables.
- `DataCollatorForSeq2Seq` dynamically pads inputs and labels to the longest sequence in each batch (more memory-efficient than static max-length padding).
- **ROUGE-L** measures the longest common subsequence between prediction and reference — higher is better, but numbers are relative to the dataset and training duration.
- With only 500 examples and 1 epoch the ROUGE score will be low; in production you'd train on the full 287 k examples for 3 + epochs.


## Step 4 · Translation with Helsinki-NLP opus-mt models

Helsinki-NLP's OPUS-MT family provides 1000+ language-pair translation models, all hosted on the Hub. The naming convention is `Helsinki-NLP/opus-mt-{src}-{tgt}` where `src` and `tgt` are ISO 639-1 codes.

| Language pair | Model |
|---|---|
| English → French | `Helsinki-NLP/opus-mt-en-fr` |
| English → Spanish | `Helsinki-NLP/opus-mt-en-es` |
| English → German | `Helsinki-NLP/opus-mt-en-de` |

These are **MarianMT** models — a fast, compact seq2seq architecture well-suited for translation.


In [ ]:
from transformers import pipeline

translator = pipeline(
    "translation_en_to_fr",
    model="Helsinki-NLP/opus-mt-en-fr",
    device=-1,
)

sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning is transforming the way we interact with technology.",
    "I would like to book a table for two at eight o'clock tonight.",
    "The train to Paris departs from platform seven in ten minutes.",
    "She studied French literature at the University of Lyon for three years.",
]

translations = translator(sentences)

for src, tgt in zip(sentences, translations):
    print(f"EN: {src}")
    print(f"FR: {tgt['translation_text']}")
    print()


### What just happened?
- The `translation_en_to_fr` task string maps to `MarianMTModel` + `MarianTokenizer` under the hood.
- MarianMT adds language-specific `>>fr<<` tokens (target language tag) automatically when using the pipeline — you don't need to add them manually.
- The pipeline batches all 5 sentences in one forward pass (default `batch_size=1`; set `batch_size=8` for throughput on GPU).
- **Production tip:** For production throughput, prefer `pipeline(..., batch_size=32)` and pass lists rather than calling the pipeline in a loop.


## Step 5 · ROUGE-L: greedy decoding vs. beam search

Beam search explores multiple hypothesis sequences in parallel, keeping the top `num_beams` candidates at each step. It consistently outperforms greedy decoding on summarization.

| Strategy | How it works | Typical ROUGE-L |
|---|---|---|
| Greedy | Always picks the highest-probability next token | Baseline |
| Beam search (k=4) | Keeps top-4 hypotheses at each step | +2–5 ROUGE pts |

We'll generate summaries with both strategies and compare ROUGE-L on our validation articles.


In [ ]:
import evaluate
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

rouge = evaluate.load("rouge")

# Reload BART-large-CNN for comparison (or use trained T5 above)
ckpt = "facebook/bart-large-cnn"
bart_tokenizer = AutoTokenizer.from_pretrained(ckpt)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(ckpt)
bart_model.eval()

# Small comparison set — 5 articles from CNN validation split
val_sample = load_dataset("cnn_dailymail", "3.0.0", split="validation[:5]")

def generate_summaries(articles, num_beams=1):
    """Generate summaries with either greedy (num_beams=1) or beam search."""
    inputs = bart_tokenizer(
        articles, return_tensors="pt",
        max_length=1024, truncation=True, padding=True
    )
    with torch.no_grad():
        outputs = bart_model.generate(
            **inputs,
            num_beams=num_beams,
            length_penalty=2.0 if num_beams > 1 else 1.0,
            early_stopping=True if num_beams > 1 else False,
            max_new_tokens=128,
        )
    return bart_tokenizer.batch_decode(outputs, skip_special_tokens=True)

articles   = val_sample["article"]
references = val_sample["highlights"]

greedy_preds = generate_summaries(articles, num_beams=1)
beam_preds   = generate_summaries(articles, num_beams=4)

greedy_rouge = rouge.compute(predictions=greedy_preds, references=references, use_stemmer=True)
beam_rouge   = rouge.compute(predictions=beam_preds,   references=references, use_stemmer=True)

print("Greedy ROUGE-L:", round(greedy_rouge["rougeL"], 4))
print("Beam-4 ROUGE-L:", round(beam_rouge["rougeL"],   4))
print(f"Improvement    : +{round(beam_rouge['rougeL'] - greedy_rouge['rougeL'], 4)}")


### What just happened?
- **Greedy decoding** (`num_beams=1`) is fast but myopic — it can lock into sub-optimal sequences early.
- **Beam search** (`num_beams=4`) keeps 4 partial hypotheses alive and picks the globally best finished sequence.
- `length_penalty=2.0` encourages the beam-search model to generate longer summaries, which typically score higher on ROUGE because they cover more reference n-grams.
- The improvement is usually 1–5 ROUGE-L points — meaningful in academic evaluation, though human judgment may not always agree.


In [ ]:
# Challenge: Translation quality comparison
# Task: Load TWO translation models (EN→FR and EN→DE) and compare their outputs
# on the same 3 sentences. Then compute a SacreBLEU score for the EN→FR model
# against a manually written French reference.
#
# Scaffold:

from transformers import pipeline
# import evaluate

sentences_en = [
    "Artificial intelligence is changing the world.",
    "I love learning new programming languages.",
    "The dataset must be split into train and test sets.",
]

# 1. Create an EN→FR pipeline with Helsinki-NLP/opus-mt-en-fr
# fr_pipe = ...

# 2. Create an EN→DE pipeline with Helsinki-NLP/opus-mt-en-de
# de_pipe = ...

# 3. Translate sentences_en with both pipelines and print side-by-side
# fr_results = ...
# de_results = ...

# 4. Load the sacrebleu metric and compute a score for fr_results
#    against these French references:
fr_references = [
    ["L'intelligence artificielle change le monde."],
    ["J'adore apprendre de nouveaux langages de programmation."],
    ["Le jeu de données doit être divisé en ensembles d'entraînement et de test."],
]
# bleu = evaluate.load('sacrebleu')
# score = bleu.compute(predictions=..., references=fr_references)
# print('SacreBLEU:', score['score'])


---
## Day 10 key concepts recap

| Concept | What to remember |
|---|---|
| Seq2Seq architecture | Encoder reads input, decoder generates output token-by-token |
| T5 task prefix | Must prepend `"summarize: "` so T5 activates the right head |
| `predict_with_generate=True` | Required in `Seq2SeqTrainingArguments` for ROUGE scoring |
| `length_penalty` | Values > 1.0 reward longer output; useful for summarization |
| Beam search vs. greedy | Beam search is slower but consistently higher ROUGE-L |
| Helsinki-NLP OPUS-MT | 1000+ language pair models; naming: `opus-mt-{src}-{tgt}` |
| `-100` in labels | Tells cross-entropy loss to ignore padding positions |

> **Tip:** `Seq2SeqTrainer` needs `predict_with_generate=True` in `Seq2SeqTrainingArguments` so that `compute_metrics` receives decoded strings instead of raw logits.

---
## What's next
**Day 11** → Push your fine-tuned model to the Hugging Face Hub, write a model card, and build a Gradio Space to demo your model publicly.

Mark Day 10 complete in your [tracker](../index.html).
